# Automated Cloud Dataset Acquisition & Google Drive Storage Pipeline
**Project:** Development and Validation of an rPPG-Based System for Non-Contact Vital Sign Monitoring  
**Storage Destination:** Google Drive (`RPPG_FYP/` on 5 TB Plan)  
**Compute Environment:** Google Colab Free Cloud Runtime  
**Hardware Constraint:** Zero Local Storage / Zero Local Workstation Strain  
**Financial Constraint:** $0.00 (Free Cloud Resources Only)  

This notebook automates the acquisition, cryptographic verification, archive extraction, and directory organization of authorized rPPG benchmark datasets (UBFC-rPPG, PURE, COHFACE, VIPL-HR, MMPD) directly into your Google Drive.

## Step 1: Official Google Drive Authentication
Mount your Google Drive using Google's secure OAuth authentication. Your credentials are never handled by any third-party script.

In [ ]:
from google.colab import drive
import os
import shutil
import json
import sys

# Official Google Colab Drive Mount
drive.mount('/content/drive')
print("Google Drive mounted successfully!")

## Step 2: Storage Quota & 5 TB Capacity Verification
Verifies total, used, and free storage to confirm your 5 TB Google AI Pro storage tier.

In [ ]:
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print("=" * 60)
print("GOOGLE DRIVE STORAGE QUOTA AUDIT:")
print(f"  Total Capacity: {total / (1024**4):.2f} TB ({total / (1024**3):.1f} GB)")
print(f"  Used Storage:   {used / (1024**4):.2f} TB ({used / (1024**3):.1f} GB)")
print(f"  Free Available: {free / (1024**4):.2f} TB ({free / (1024**3):.1f} GB)")
print("=" * 60)

if (free / (1024**4)) >= 1.0:
    print("Storage Verification PASSED: Sufficient space (> 1 TB) for all rPPG datasets.")
else:
    print("Storage Warning: Free space is under 1 TB. Review contents.")

## Step 3: Establish Standard Project Hierarchy on Google Drive
Creates the standardized folder hierarchy under `RPPG_FYP/` on your Google Drive.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/RPPG_FYP"

FOLDERS = [
    "01_RAW_DATASETS",
    "02_DATASET_METADATA",
    "03_PROCESSED_DATASETS",
    "04_TRAINING_DATA",
    "05_MODELS",
    "06_EXPERIMENTS",
    "07_VALIDATION_RESULTS",
    "08_DOCUMENTATION",
]

DATASET_NAMES = ["UBFC-rPPG", "PURE", "COHFACE", "VIPL-HR", "MMPD", "BP4D_PLUS"]

for f in FOLDERS:
    p = os.path.join(DRIVE_ROOT, f)
    os.makedirs(p, exist_ok=True)

# Create individual raw dataset folders
for ds in DATASET_NAMES:
    p = os.path.join(DRIVE_ROOT, "01_RAW_DATASETS", ds)
    os.makedirs(p, exist_ok=True)

print(f"Standard RPPG_FYP hierarchy established at: {DRIVE_ROOT}")
!ls -lh "/content/drive/MyDrive/RPPG_FYP/01_RAW_DATASETS"

## Step 4: Clone Acquisition Framework & Tooling

In [ ]:
%cd /content
!git clone https://github.com/your-username/rppg-vital-sign-monitor.git || (cd rppg-vital-sign-monitor && git pull)
sys.path.insert(0, '/content/rppg-vital-sign-monitor/backend')

# Install lightweight cloud utilities
!pip install -q gdown kaggle opencv-python-headless huggingface-hub
print("Cloud acquisition environment ready.")

## Step 5: Pre-Download Inspection & Authorization Summary
Audits the status of each target dataset in your Google Drive before initiating downloads.

In [ ]:
from app.datasets.cloud_acquisition_pipeline import CloudDatasetAcquisitionEngine
import pandas as pd

engine = CloudDatasetAcquisitionEngine(drive_root=DRIVE_ROOT)

print("CURRENT DATASET STATUS ON GOOGLE DRIVE:")
print("-" * 80)
status_records = []
for ds in DATASET_NAMES:
    ds_dir = os.path.join(DRIVE_ROOT, "01_RAW_DATASETS", ds)
    audit = engine.audit_downloaded_dataset(ds_dir)
    status_records.append({
        "Dataset": ds,
        "Found On Drive": audit["exists"],
        "Size (MB)": audit.get("total_size_mb", 0.0),
        "Videos Count": audit.get("video_count", 0),
        "Integrity": audit.get("integrity_status", "NOT_DOWNLOADED"),
    })

df_status = pd.DataFrame(status_records)
display(df_status)

## Step 6: Automated Download of Publicly Authorized Datasets (MMPD)
MMPD (NeurIPS 2023) is released under the MIT Open Source License on GitHub / HuggingFace. This cell clones the repository directly into your Google Drive raw dataset folder.

In [ ]:
mmpd_dir = os.path.join(DRIVE_ROOT, "01_RAW_DATASETS", "MMPD")

if not os.path.exists(os.path.join(mmpd_dir, ".git")):
    print("Downloading MMPD dataset metadata and processing scripts from official repository...")
    !git clone --depth 1 https://github.com/McDuff-App/MMPD.git "{mmpd_dir}"
    print("MMPD base acquired into Google Drive successfully!")
else:
    print("MMPD already present in Google Drive. Skipping duplicate clone.")

## Step 7: Authorized Kaggle Dataset Downloader (Optional with Kaggle API)
For datasets mirrored on Kaggle (e.g. public mirrors of UBFC-rPPG), authenticate via Colab Secrets or upload your `kaggle.json`.

In [ ]:
# If using Kaggle API credentials:
# 1. Place kaggle.json in ~/.kaggle/kaggle.json
# 2. Example command to download directly to Drive raw folder:
# !kaggle datasets download -d <dataset-slug> -p "{DRIVE_ROOT}/01_RAW_DATASETS/UBFC-rPPG/" --unzip
print("Kaggle API integration cell ready. Mount API key if downloading Kaggle-hosted mirrors.")

## Step 8: Post-Acquisition Video Integrity & Frame Decodability Audit
Scans all downloaded video files across all Google Drive folders, testing OpenCV video container readability and counting valid decodable frames.

In [ ]:
print("RUNNING OPENCV FRAME DECODABILITY & INTEGRITY AUDIT:")
print("=" * 70)
final_audits = {}
for ds in DATASET_NAMES:
    ds_dir = os.path.join(DRIVE_ROOT, "01_RAW_DATASETS", ds)
    res = engine.audit_downloaded_dataset(ds_dir)
    final_audits[ds] = res
    print(f"Dataset: {ds:<12} | Videos: {res.get('video_count', 0):<4} | Valid: {res.get('valid_videos_count', 0):<4} | Status: {res.get('integrity_status')}")

# Export audit report to 02_DATASET_METADATA
report_path = os.path.join(DRIVE_ROOT, "02_DATASET_METADATA", "download_verification_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(final_audits, f, indent=2)
print(f"\nFull verification report saved to: {report_path}")

## Step 9: Synchronize Master Dataset Inventory CSV
Copies the verified master `Dataset_Inventory.csv` into `02_DATASET_METADATA/` on your Google Drive.

In [ ]:
src_csv = "/content/rppg-vital-sign-monitor/Dataset_Inventory.csv"
dest_csv = os.path.join(DRIVE_ROOT, "02_DATASET_METADATA", "Dataset_Inventory.csv")

if os.path.exists(src_csv):
    shutil.copy2(src_csv, dest_csv)
    print(f"Master inventory CSV synced to Google Drive: {dest_csv}")
    df_inv = pd.read_csv(dest_csv)
    display(df_inv[["Dataset Name", "Access Status", "HR Availability", "RR Availability", "SpO2 Availability", "BP Availability", "Estimated Size"]])
else:
    print("Local inventory file not found. Check git clone status.")